# Pipeline pydeseq su les données de Rnaseq

## Importation des packages

In [83]:
import os
import pickle as pkl

import pandas as pd

from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats


## Fichier count

In [84]:
import pydeseq2
print(pydeseq2.__version__)

0.5.2


## Lecture du fichier

La première commande consiste à lire le fichier de feature counts, en ignorant les lignes qui sont en commentaires. Le séparateurs de ce fichier est la tabluation.

In [108]:
counts=pd.read_csv("raw_counts.txt", sep='\t', comment="#")
counts

,Geneid,Chr,Start,End,Strand,Length,../results/RNAseq1-KO1_S13_Aligned.sortedByCoord.out.bam,../results/RNAseq1-KO2_S14_Aligned.sortedByCoord.out.bam,../results/RNAseq1-KO3_S15_Aligned.sortedByCoord.out.bam,../results/RNAseq1-KO4_S16_Aligned.sortedByCoord.out.bam,...,../results/RNAseq2-WT3_S27_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WT4_S28_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WT5_S29_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WT6_S30_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind1_S31_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind2_S32_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind3_S33_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind4_S34_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind5_S35_Aligned.sortedByCoord.out.bam,../results/RNAseq2-WTind6_S36_Aligned.sortedByCoord.out.bam
0,ENSG00000290825.2,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;...,11121;11125;11410;11411;11426;11770;11819;1182...,11211;11211;11671;11671;11671;11844;11844;1184...,+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;+;...,2898,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ENSG00000223972.6,1;1;1;1;1;1,12010;12179;12613;12975;13221;13453,12057;12227;12697;13052;13374;13670,+;+;+;+;+;+,632,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000310526.1,1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;1;...,14356;14356;14356;14356;14356;14356;14356;1436...,14829;14829;14829;14829;14829;14829;14829;1482...,-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;-;...,6775,235,248,178,207,...,178,209,109,164,128,203,167,176,189,215
3,ENSG00000227232.6,1;1;1;1;1;1;1;1;1;1,14696;14970;15796;16607;16858;17233;17606;1791...,14829;15038;15947;16765;17055;17368;17742;1806...,-;-;-;-;-;-;-;-;-;-,1380,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ENSG00000278267.1,1,17369,17436,-,68,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78686,ENSG00000198695.2,M,14149,14673,-,525,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78687,ENSG00000210194.1,M,14674,14742,-,69,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78688,ENSG00000198727.2,M,14747,15887,+,1141,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78689,ENSG00000210195.2,M,15888,15953,+,66,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Suppression des colonnes Chr, Start, End, Strand, Length.
Modifications des noms des gènes et des échantillons.

In [ ]:
counts.drop(["Chr","Start","End","Strand","Length"], axis=1, inplace=True)
counts["Geneid"]=counts["Geneid"].str.split(".").str[0]
counts.columns=counts.columns.str.replace("../results/", "").str.replace("_Aligned.sortedByCoord.out.bam", "")
counts

,Geneid,RNAseq1-KO1_S13,RNAseq1-KO2_S14,RNAseq1-KO3_S15,RNAseq1-KO4_S16,RNAseq1-KO5_S17,RNAseq1-KO6_S18,RNAseq1-KOind1_S19,RNAseq1-KOind2_S20,RNAseq1-KOind3_S21,...,RNAseq2-WT3_S27,RNAseq2-WT4_S28,RNAseq2-WT5_S29,RNAseq2-WT6_S30,RNAseq2-WTind1_S31,RNAseq2-WTind2_S32,RNAseq2-WTind3_S33,RNAseq2-WTind4_S34,RNAseq2-WTind5_S35,RNAseq2-WTind6_S36
0,ENSG00000290825,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,ENSG00000223972,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000310526,235,248,178,207,0,405,205,281,248,...,178,209,109,164,128,203,167,176,189,215
3,ENSG00000227232,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ENSG00000278267,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78686,ENSG00000198695,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78687,ENSG00000210194,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78688,ENSG00000198727,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78689,ENSG00000210195,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


On retire les outliers identifiés lors du contrôles qualités

In [ ]:
outlier = ["RNAseq1-KO5_S17","RNAseq2-KOind2_S44","RNAseq2-KOind4_S46"]
counts = counts.drop(columns=outlier)
counts

,Geneid,RNAseq1-KO1_S13,RNAseq1-KO2_S14,RNAseq1-KO3_S15,RNAseq1-KO4_S16,RNAseq1-KO6_S18,RNAseq1-KOind1_S19,RNAseq1-KOind2_S20,RNAseq1-KOind3_S21,RNAseq1-KOind4_S22,...,RNAseq2-WT3_S27,RNAseq2-WT4_S28,RNAseq2-WT5_S29,RNAseq2-WT6_S30,RNAseq2-WTind1_S31,RNAseq2-WTind2_S32,RNAseq2-WTind3_S33,RNAseq2-WTind4_S34,RNAseq2-WTind5_S35,RNAseq2-WTind6_S36
0,ENSG00000290825,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,ENSG00000223972,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENSG00000310526,235,248,178,207,405,205,281,248,103,...,178,209,109,164,128,203,167,176,189,215
3,ENSG00000227232,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ENSG00000278267,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78686,ENSG00000198695,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78687,ENSG00000210194,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78688,ENSG00000198727,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
78689,ENSG00000210195,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


On fait la transposition des gènes avec les échantillons, et on met Geneid en index. Filtration des gènes dont la somme de l'expression est inférieur à 10.

In [111]:
counts=counts.set_index("Geneid")
counts=counts.T
counts.index.name=None
counts=counts.loc[:,counts.sum(axis=0) >= 10]
counts

Geneid,ENSG00000310526,ENSG00000243485,ENSG00000308361,ENSG00000241860,ENSG00000233750,ENSG00000308579,ENSG00000268903,ENSG00000269981,ENSG00000239906,ENSG00000310528,...,ENSG00000291033,ENSG00000267793,ENSG00000260197,ENSG00000012817,ENSG00000288049,ENSG00000229236,ENSG00000277438,ENSG00000198692,ENSG00000280969,ENSG00000237917
RNAseq1-KO1_S13,235,1,0,29,60,115,4,161,0,3,...,4346,23,75,5667,46,14,0,2117,6,4
RNAseq1-KO2_S14,248,1,0,10,41,123,2,261,2,2,...,4792,38,77,6585,45,21,4,2277,2,14
RNAseq1-KO3_S15,178,6,0,16,47,138,7,201,1,1,...,4534,31,61,6069,20,17,0,2337,2,6
RNAseq1-KO4_S16,207,2,0,12,31,139,4,186,1,0,...,4290,39,54,4672,42,8,0,1735,0,6
RNAseq1-KO6_S18,405,0,2,1,40,392,3,271,0,0,...,3058,4,101,4694,9,14,0,5514,8,12
RNAseq1-KOind1_S19,205,1,0,16,28,254,6,410,2,0,...,4550,65,72,7139,56,22,0,2622,0,4
RNAseq1-KOind2_S20,281,1,0,27,26,413,7,560,3,8,...,6694,81,88,10559,64,34,0,4092,2,12
RNAseq1-KOind3_S21,248,2,0,19,30,453,9,566,4,3,...,5527,59,93,7226,26,18,0,3217,0,6
RNAseq1-KOind4_S22,103,0,0,3,16,178,8,226,2,4,...,2704,21,35,4365,37,14,0,1597,0,2
RNAseq1-KOind5_S23,209,1,0,13,25,372,7,498,7,7,...,5051,41,66,7290,61,27,0,3106,0,8


Sauvegarde du fichier filtrer qui à servie à l'analyse

In [ ]:
counts.to_csv("final_counts.csv")

## Metadata

Construction du métadata

In [91]:
metadata_df = counts.index.to_frame(index=False)
metadata_df.columns = ["sample"]

In [92]:
metadata_df

,sample
0,RNAseq1-KO1_S13
1,RNAseq1-KO2_S14
2,RNAseq1-KO3_S15
3,RNAseq1-KO4_S16
4,RNAseq1-KO6_S18
5,RNAseq1-KOind1_S19
6,RNAseq1-KOind2_S20
7,RNAseq1-KOind3_S21
8,RNAseq1-KOind4_S22
9,RNAseq1-KOind5_S23


In [93]:
genotype=[]
for i in metadata_df["sample"]:
    if "KO" in i :
        genotype.append("KO")
    elif "WT" in i:
        genotype.append("WT")
metadata_df["Genotype"]=genotype

In [94]:
induction=[]
for i in metadata_df["sample"]:
    if "ind" in i :
        induction.append("induced")
    else:
        induction.append("uninduced")
metadata_df["Induction"]=induction

In [95]:
condition=[]
for i in metadata_df["sample"]:
    if "KOind" in i :
        condition.append("KO_induced")
    elif "KO" in i : 
        condition.append("KO")
    elif "WTind" in i : 
        condition.append("WT_induced")
    else:
        condition.append("WT")
metadata_df["Condition"]=condition

In [96]:
batch=[]
for i in metadata_df["sample"]:
    if "RNAseq1" in i :
        batch.append("group1")
    else:
        batch.append("group2")
metadata_df["Batch"]=batch

In [97]:
metadata_df


,sample,Genotype,Induction,Condition,Batch
0,RNAseq1-KO1_S13,KO,uninduced,KO,group1
1,RNAseq1-KO2_S14,KO,uninduced,KO,group1
2,RNAseq1-KO3_S15,KO,uninduced,KO,group1
3,RNAseq1-KO4_S16,KO,uninduced,KO,group1
4,RNAseq1-KO6_S18,KO,uninduced,KO,group1
5,RNAseq1-KOind1_S19,KO,induced,KO_induced,group1
6,RNAseq1-KOind2_S20,KO,induced,KO_induced,group1
7,RNAseq1-KOind3_S21,KO,induced,KO_induced,group1
8,RNAseq1-KOind4_S22,KO,induced,KO_induced,group1
9,RNAseq1-KOind5_S23,KO,induced,KO_induced,group1


Mettre sample en index

In [98]:
metadata_df = metadata_df.set_index("sample")

In [99]:
metadata_df = metadata_df.loc[counts.index]

Sauvegarde du metadata

In [100]:
metadata_df.to_csv("metadata.csv")

## Pydeseq2

In [101]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=counts,
    metadata=metadata_df,
    design="~Condition",
    refit_cooks=True,
    inference=inference,
)

In [102]:
dds.deseq2()

Fitting size factors...
... done in 0.06 seconds.



Using None as control genes, passed at DeseqDataSet initialization


/home/rademe/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/rademe/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/rademe/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/rademe/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
/

In [103]:
print(dds)

AnnData object with n_obs × n_vars = 44 × 46771
    obs: 'Genotype', 'Induction', 'Condition', 'Batch', 'size_factors', 'replaceable'
    var: '_normed_means', 'non_zero', '_MoM_dispersions', 'genewise_dispersions', '_genewise_converged', 'fitted_dispersions', 'MAP_dispersions', '_MAP_converged', 'dispersions', '_outlier_genes', '_LFC_converged', 'replaced', 'refitted', '_pvalue_cooks_outlier'
    uns: 'trend_coeffs', 'disp_function_type', '_squared_logres', 'prior_disp_var'
    obsm: 'design_matrix', '_mu_LFC', '_hat_diagonals'
    varm: 'LFC'
    layers: 'normed_counts', '_mu_hat', 'cooks', 'replace_cooks'


In [104]:
print(dds.var["dispersions"])

Geneid
ENSG00000310526     0.050992
ENSG00000243485     0.969923
ENSG00000308361    15.545501
ENSG00000241860     0.246637
ENSG00000233750     0.103521
                     ...    
ENSG00000229236     0.112332
ENSG00000277438    23.582978
ENSG00000198692     0.054532
ENSG00000280969     3.304221
ENSG00000237917     0.369989
Name: dispersions, Length: 46771, dtype: float64


In [105]:
print(dds.varm["LFC"])


                 Intercept  Condition[T.KO_induced]  Condition[T.WT]  \
Geneid                                                                 
ENSG00000310526   5.465932                -0.117077        -0.213434   
ENSG00000243485   0.784880                -0.719054        -0.150808   
ENSG00000308361  -1.299915                 0.122471         0.328758   
ENSG00000241860   2.912084                -0.185237        -0.015834   
ENSG00000233750   3.397448                -0.418272         0.274270   
...                    ...                      ...              ...   
ENSG00000229236   2.725067                 0.286572        -0.436967   
ENSG00000277438  -0.383237                -1.238649        -0.993269   
ENSG00000198692   7.571328                 0.276389         0.229817   
ENSG00000280969   0.238098                -0.379182        -0.192560   
ENSG00000237917   2.006701                -0.030768         0.305502   

                 Condition[T.WT_induced]  
Geneid              

Création des statistiques pour chacune des conditions

In [106]:
ds_WT_induced = DeseqStats(dds, contrast=["Condition", "WT_induced", "WT"], inference=inference)
ds_KO_induced = DeseqStats(dds, contrast=["Condition", "KO_induced", "KO"], inference=inference)
ds_KO_WT_induced = DeseqStats(dds, contrast=["Condition", "KO_induced", "WT_induced"], inference=inference)
ds_KO_WT = DeseqStats(dds, contrast=["Condition", "KO", "WT"], inference=inference)

In [107]:
ds_WT_induced.summary()
ds_KO_induced.summary()
ds_KO_WT_induced.summary()
ds_KO_WT.summary()



Running Wald tests...
... done in 2.45 seconds.



Log2 fold change & Wald test p-value: Condition WT_induced vs WT
                    baseMean  log2FoldChange     lfcSE      stat  \
Geneid                                                             
ENSG00000310526   219.713955        0.308297  0.139420  2.211283   
ENSG00000243485     1.872687       -0.046795  0.725226 -0.064525   
ENSG00000308361     0.217830       -0.134187  2.533741 -0.052960   
ENSG00000241860    16.567196       -0.341136  0.329656 -1.034824   
ENSG00000233750    24.751765       -1.991675  0.244041 -8.161239   
...                      ...             ...       ...       ...   
ENSG00000229236    14.288245        0.433677  0.262055  1.654907   
ENSG00000277438     0.226295       -0.273186  3.132203 -0.087219   
ENSG00000198692  2356.485383        0.031028  0.138102  0.224672   
ENSG00000280969     0.931795       -0.444600  1.248690 -0.356053   
ENSG00000237917     8.107367       -0.608441  0.417667 -1.456760   

                       pvalue          padj  
Gene

Running Wald tests...
... done in 2.52 seconds.



Log2 fold change & Wald test p-value: Condition KO_induced vs KO
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Geneid                                                                       
ENSG00000310526   219.713955       -0.168907  0.153689 -1.099014  0.271762   
ENSG00000243485     1.872687       -1.037376  0.854472 -1.214055  0.224727   
ENSG00000308361     0.217830        0.176688  2.854886  0.061890  0.950651   
ENSG00000241860    16.567196       -0.267241  0.363419 -0.735352  0.462125   
ENSG00000233750    24.751765       -0.603439  0.251602 -2.398385  0.016468   
...                      ...             ...       ...       ...       ...   
ENSG00000229236    14.288245        0.413436  0.269983  1.531338  0.125686   
ENSG00000277438     0.226295       -1.786992  3.401025 -0.525428  0.599286   
ENSG00000198692  2356.485383        0.398745  0.152143  2.620852  0.008771   
ENSG00000280969     0.931795       -0.547044  1.360130 -0.402200  0.687537   

Running Wald tests...
... done in 2.26 seconds.



Log2 fold change & Wald test p-value: Condition KO_induced vs WT_induced
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Geneid                                                                       
ENSG00000310526   219.713955       -0.169284  0.151148 -1.119992  0.262717   
ENSG00000243485     1.872687       -0.773010  0.856159 -0.902882  0.366588   
ENSG00000308361     0.217830       -0.163422  2.786397 -0.058650  0.953231   
ENSG00000241860    16.567196        0.096739  0.362621  0.266779  0.789640   
ENSG00000233750    24.751765        0.992548  0.273050  3.635037  0.000278   
...                      ...             ...       ...       ...       ...   
ENSG00000229236    14.288245        0.610169  0.270172  2.258449  0.023918   
ENSG00000277438     0.226295       -0.080822  3.448491 -0.023437  0.981302   
ENSG00000198692  2356.485383        0.036162  0.149223  0.242334  0.808521   
ENSG00000280969     0.931795        0.175362  1.378814  0.127183  0.8

Running Wald tests...
... done in 2.27 seconds.



Log2 fold change & Wald test p-value: Condition KO vs WT
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
Geneid                                                                       
ENSG00000310526   219.713955        0.307920  0.142171  2.165835  0.030324   
ENSG00000243485     1.872687        0.217570  0.723234  0.300829  0.763545   
ENSG00000308361     0.217830       -0.474298  2.608873 -0.181802  0.855738   
ENSG00000241860    16.567196        0.022844  0.330535  0.069112  0.944900   
ENSG00000233750    24.751765       -0.395688  0.219779 -1.800386  0.071800   
...                      ...             ...       ...       ...       ...   
ENSG00000229236    14.288245        0.630410  0.261861  2.407425  0.016065   
ENSG00000277438     0.226295        1.432984  3.079870  0.465274  0.641735   
ENSG00000198692  2356.485383       -0.331556  0.141252 -2.347259  0.018912   
ENSG00000280969     0.931795        0.277806  1.228028  0.226221  0.821029   
ENSG000

Sauvegarde de chaque fichier

In [113]:
ds_WT_induced.results_df.to_csv("results_WT_induced.csv")
ds_KO_WT_induced.results_df.to_csv("results_KO_WT_induced.csv")
ds_KO_WT.results_df.to_csv("results_KO_WT.csv")
ds_KO_induced.results_df.to_csv("results_KO_induced.csv")